# Evaluating a trained greyscale LNP model — Hoefling et al., 2024

Greyscale counterpart of `eval_hoefling_2024_lnp.ipynb`. It loads an **already trained**
linear-nonlinear-Poisson model whose core collapses the two stimulus colour channels (green, UV)
to their mean before the readout, so each neuron carries 1 x 18 x 16 = 288 weights instead of 576.
Nothing is trained here.

The model is downloaded from the open-retina HuggingFace repo by name, so this notebook is
self-contained — nothing needs to exist on your machine beforehand.

| | value |
|---|---|
| `model.core.color_squashing_weights` | `[0.5, 0.5]` |
| `readout.smooth_weight` | `3e4` (the shipped default of `1.0` is numerically inert) |
| `readout.sparse_weight` | `1` |
| best val correlation | 0.0925 at epoch 51 |
| test correlation (trial-averaged) | 0.2563 |

How the four arms compare:

| model | val corr | test corr |
|---|---|---|
| grey, `smooth_weight=1` | 0.0926 | 0.2430 |
| grey, `smooth_weight=3e4` | 0.0925 | **0.2563** |
| colour, `smooth_weight=1` | 0.0860 | 0.2503 |
| colour, `smooth_weight=3e4` | 0.0983 | 0.2594 |

**Read the test column, not the val column.** Validation correlation here is single-trial and
noise-dominated: the two greyscale arms are indistinguishable on it (0.0926 vs 0.0925) yet differ
by 0.013 on the trial-averaged test set. Both `EarlyStopping` and `ModelCheckpoint` monitor the val
metric, so model selection cannot separate them. See `hoefling_2024_lnp_greyscale_plan.md`.

Data still has to be loaded — the stimuli and responses are what we evaluate against — but the
training, callbacks, logging and TensorBoard sections are all gone.

## What you need to run this

Nothing but a clone of this repo and its environment. Everything else downloads on first use into
the openretina cache (`$OPENRETINA_CACHE_DIRECTORY`, or `~/openretina_cache` if unset):

| | |
|---|---|
| model checkpoint | ~15 MB, pulled from HuggingFace by name |
| stimuli + responses | ~1.9 GB, pulled on first run and cached |
| peak RAM | ~5 GB (the 67-session response load dominates) |
| GPU | not needed — this model is <1 M parameters and runs fine on CPU |
| runtime | a few minutes, most of it the first-time download |

# Imports

In [ ]:
import logging
import os

import hydra
import lightning
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from einops import rearrange

from openretina.data_io.base import compute_data_info
from openretina.data_io.cyclers import ShortCycler
from openretina.data_io.hoefling_2024.dataloaders import natmov_dataloaders_v2
from openretina.data_io.hoefling_2024.responses import filter_responses, make_final_responses
from openretina.data_io.hoefling_2024.stimuli import movies_from_pickle
from openretina.eval.metrics import correlation_numpy, feve
from openretina.models.core_readout import load_core_readout_model
from openretina.utils.file_utils import get_cache_directory, get_local_file_path
from openretina.utils.h5_handling import load_h5_into_dict
from openretina.utils.misc import CustomPrettyPrinter

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

pp = CustomPrettyPrinter(indent=4, max_lines=30)

## Memory check — run this first

Loading all 67 sessions peaks at about **4.5 GB**. If you are inside a memory-capped allocation
(SLURM, a container, a cgroup-limited VM) and the cap is exceeded, the kernel is killed *silently*
— no traceback, just a dead kernel. This cell reports the cap up front so that failure mode is
legible instead of mysterious.

It is a no-op on an ordinary workstation: with no cgroup limit set it simply says so and continues.
If it does report a cap that is too small, restart the kernel inside a larger allocation — for
SLURM that means a job with at least `--mem=8G`.

In [ ]:
from pathlib import Path


def cgroup_memory_limit_gb() -> float | None:
    """Effective memory ceiling of the enclosing cgroup, or None if unlimited/unknown.

    Under SLURM the cap sits on an ancestor cgroup (the job scope), not on the leaf the process
    lives in, so we take the minimum over the whole chain up to the root.
    """
    try:
        rel = next(
            line.split(":", 2)[2].strip()
            for line in open("/proc/self/cgroup")
            if line.startswith("0::")
        )
    except (OSError, StopIteration):
        try:  # cgroup v1
            value = int(open("/sys/fs/cgroup/memory/memory.limit_in_bytes").read())
            return value / 1024**3 if value < 2**62 else None
        except OSError:
            return None

    root, path, limits = Path("/sys/fs/cgroup"), Path("/sys/fs/cgroup") / rel.lstrip("/"), []
    while True:
        try:
            raw = (path / "memory.max").read_text().strip()
            if raw != "max":
                limits.append(int(raw))
        except OSError:
            pass
        if path == root:
            break
        path = path.parent
    return min(limits) / 1024**3 if limits else None


NEEDED_GB = 4.5  # peak RSS of the full 67-session load, measured on the batch runs

limit_gb = cgroup_memory_limit_gb()
if limit_gb is None:
    print("no cgroup memory cap found; assuming this is not a capped allocation")
else:
    print(f"allocation memory cap: {limit_gb:.1f} GB (this notebook needs ~{NEEDED_GB} GB)")
    if limit_gb < NEEDED_GB * 1.5:
        raise MemoryError(
            f"This allocation caps memory at {limit_gb:.1f} GB, but loading all sessions peaks at "
            f"~{NEEDED_GB} GB. The kernel would be killed without a traceback part-way through the "
            f"data cells. Relaunch the VS Code job with --mem=32G (see the cell above), or run the "
            f"evaluation as a batch job instead."
        )


# Config and paths

We compose the same config the model was trained with, so the data loading matches exactly.

`OPENRETINA_CACHE_DIRECTORY` decides where the ~1.9 GB of stimuli and responses land. Set it
before starting the kernel if you want them somewhere other than `~/openretina_cache` — on a
cluster that usually means shared scratch rather than your home directory.

In [ ]:
with hydra.initialize(config_path=os.path.join("..", "configs"), version_base="1.3"):
    cfg = hydra.compose(config_name="hoefling_2024_core_readout_low_res_lnp_grey.yaml")

CACHE_DIR = os.environ.get("OPENRETINA_CACHE_DIRECTORY") or get_cache_directory()
cfg.paths.cache_dir = CACHE_DIR
os.environ["OPENRETINA_CACHE_DIRECTORY"] = CACHE_DIR

# `paths.output_dir` defaults to `${hydra:runtime.output_dir}`, which only resolves inside a hydra
# job. We compose by hand here, so point it somewhere concrete.
cfg.paths.log_dir = "."
cfg.paths.output_dir = "."

print(f"cache: {CACHE_DIR}")

# Data

Same three steps as in the training notebook: stimuli, responses, dataloaders. This is the slow
part of the notebook (a couple of minutes and a few GB of RAM for all 67 sessions).

In [ ]:
movies_path = get_local_file_path(file_path=cfg.paths.movies_path, cache_folder=cfg.paths.data_dir)
movies_dict = movies_from_pickle(movies_path)

pp.pprint(movies_dict)

In [ ]:
responses_path = get_local_file_path(file_path=cfg.paths.responses_path, cache_folder=cfg.paths.data_dir)

responses_dict = load_h5_into_dict(file_path=responses_path)
filtered_responses_dict = filter_responses(responses_dict, **cfg.quality_checks)
final_responses = make_final_responses(filtered_responses_dict, response_type="natural")

print(f"{len(final_responses)} sessions")

In [ ]:
dataloaders = natmov_dataloaders_v2(
    neuron_data_dictionary=final_responses,
    movies_dictionary=movies_dict,
    allow_over_boundaries=True,
    batch_size=cfg.dataloader.batch_size,
    train_chunk_size=cfg.dataloader.train_chunk_size,
    validation_clip_indices=cfg.dataloader.validation_clip_indices,
)

data_info = compute_data_info(neuron_data_dictionary=final_responses, movies_dictionary=movies_dict)

print(f"{sum(data_info['n_neurons_dict'].values())} neurons over {len(data_info['n_neurons_dict'])} sessions")

# Loading the trained model

`load_core_readout_model` restores a `UnifiedCoreReadout` (weights, hyperparameters and the
`data_info` recorded at training time). It accepts the name of a published model, a local path, or
a path relative to the openretina cache. We use the published name here so this notebook runs
anywhere.

In [ ]:
# `load_core_readout_model` takes either a published model name or a path. The name resolves
# through `_MODEL_NAME_TO_REMOTE_LOCATION` (openretina/models/core_readout.py) and downloads the
# checkpoint into the openretina cache on first use.
MODEL = "hoefling_2024_lnp_low_res_grey_scale"

# To evaluate a local training run instead, point MODEL at its `_final` checkpoint — that is the
# one written after `trainer.test`, so it carries the test-time `data_info` as well as the weights:
# MODEL = ("../openretina_assets/runs/lnp_grey/<jobid>/grey_smooth3e4/checkpoints/"
#          "epoch=51_val_evaluation_loss=0.093_final.ckpt")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_core_readout_model(MODEL, device)
model.eval()

example_readout = model.readout[next(iter(model.readout.keys()))]
print(f"{type(model).__name__} on {device}")
print(f"  readout              {type(model.readout).__name__}, {len(model.readout.keys())} sessions")
print(f"  smooth / sparse      {example_readout.smooth_weight} / {example_readout.sparse_weight}")
print(f"  learning rate        {model.learning_rate}")
print(f"  trainable parameters {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Is this actually a greyscale model?

Worth checking explicitly. The readout's channel count is *derived* by probing the core at
construction time, never declared in the config, so the only way to confirm from a checkpoint
alone that the squash really happened is to look at the core layer and the readout kernel.

In [ ]:
squash = model.core.color_squashing_layer
kernel = example_readout.inner_product_kernel.weight

print(f"core squashing layer  {squash}")
print(f"channel weights       {None if squash is None else squash.channel_weights.tolist()}")
print(f"readout in_channels   {example_readout.in_channels}")
print(f"readout kernel        {tuple(kernel.shape)}  ->  {kernel[0].numel()} weights/neuron")
print(f"stimulus it expects   {model.stimulus_shape(time_steps=150)}  (still 2-channel: the squash is internal)")

assert squash is not None, "this checkpoint has no colour squashing - it is a COLOUR model"
assert example_readout.in_channels == 1, f"expected 1 readout channel, got {example_readout.in_channels}"
assert kernel[0].numel() == 18 * 16, "unexpected kernel size"
print("\nconfirmed greyscale: 288 weights/neuron, half the colour model")


What the model recorded about itself at training time:

In [ ]:
for key in ("pretrained_performance_metric", "pretrained_performance", "model_cut_frames", "input_shape"):
    if key in model.data_info:
        print(f"{key:32s} {model.data_info[key]}")

A checkpoint only makes sense against the sessions it was trained on — the readout is
session-specific. Check that the loaded model and the data we just built agree:

In [ ]:
model_sessions = set(model.readout.keys())
data_sessions = set(data_info["n_neurons_dict"].keys())

print(f"model {len(model_sessions)} sessions, data {len(data_sessions)} sessions")
print(f"only in model: {sorted(model_sessions - data_sessions)}")
print(f"only in data:  {sorted(data_sessions - model_sessions)}")
assert model_sessions == data_sessions, "checkpoint and data do not describe the same sessions"

mismatched = {
    k: (model.readout[k].number_of_neurons(), data_info["n_neurons_dict"][k])
    for k in sorted(model_sessions)
    if model.readout[k].number_of_neurons() != data_info["n_neurons_dict"][k]
}
print(f"neuron-count mismatches: {mismatched if mismatched else 'none'}")

# Quantitative evaluation

The same numbers `openretina train` prints at the end of training: Poisson loss and correlation on
each split. `CorrelationLoss3d` is the average correlation per neuron (higher is better), and is
what early stopping and the LR scheduler monitor during training.

In [ ]:
loaders = {name: ShortCycler(dataloaders[name]) for name in ("train", "validation", "test")}
print({f"DataLoader {i}": name for i, name in enumerate(loaders)})

# logger=False / enable_checkpointing=False: we are only reading metrics out, and without a
# checkpoint callback the model's `on_test_end` hook will not write a stray `_final.ckpt`.
trainer = lightning.Trainer(logger=False, enable_checkpointing=False, accelerator="auto", devices=1)
test_results = trainer.test(model, dataloaders=list(loaders.values()))

# Fraction of explainable variance explained

Correlation does not tell us how much of the *explainable* (i.e. non-noise) variance the model
captures. FEVe does, using the repeated presentations of the test clip.

In [ ]:
example_session = sorted(final_responses.keys())[1]

responses_by_trial = final_responses[example_session].test_by_trial  # (neurons, time, trials)
test_movie = dataloaders["test"][example_session].dataset.movies

with torch.no_grad():
    model_predictions = model(test_movie.to(model.device).unsqueeze(0), data_key=example_session)
model_predictions = model_predictions.squeeze(0).cpu().numpy()  # (time, neurons)

# The core drops the first frames, so the targets have to be trimmed by the same amount.
lag = responses_by_trial.shape[1] - model_predictions.shape[0]
print(f"session {example_session}: {responses_by_trial.shape} responses, "
      f"{model_predictions.shape} predictions, lag {lag} frames")

feve_score = feve(
    rearrange(responses_by_trial, "neurons time trials -> time trials neurons")[lag:],
    model_predictions,
)
correlations = correlation_numpy(
    final_responses[example_session].test_response.T[lag:], model_predictions, axis=0
)

print(f"mean FEVe        {feve_score.mean():.3f}")
print(f"mean correlation {correlations.mean():.3f}")

In [ ]:
fig, (ax_feve, ax_corr) = plt.subplots(ncols=2, figsize=(11, 3.5))
ax_feve.hist(feve_score, bins=20)
ax_feve.set(xlabel="FEVe", ylabel="neurons", title=f"FEVe (mean {feve_score.mean():.3f})")
ax_corr.hist(correlations, bins=20)
ax_corr.set(xlabel="correlation", title=f"correlation (mean {correlations.mean():.3f})")
fig.suptitle(f"{example_session} — {len(feve_score)} neurons")
sns.despine(fig=fig)

# Best-predicted neuron

Prediction against the trial-averaged response for the neuron the model does best on.

In [ ]:
neuron_idx = int(np.argsort(feve_score)[-4])

mean_test_responses = final_responses[example_session].test_response  # (neurons, time)
window = model_predictions.shape[0]

plt.figure(figsize=(11, 4))
plt.plot(np.arange(lag, lag + window), mean_test_responses[neuron_idx, lag:][:window], label="target")
plt.plot(np.arange(lag, lag + window), model_predictions[:window, neuron_idx], label="prediction")
plt.xlabel("frame")
plt.ylabel("response")
plt.title(
    f"{example_session} neuron {neuron_idx} — "
    f"FEVe {feve_score[neuron_idx]:.2f}, correlation {correlations[neuron_idx]:.2f}"
)
plt.legend()
sns.despine()

# What the model actually learned

For an LNP the whole model *is* the readout: `nn.Conv3d(kernel_size=(1, 18, 16))`. In the greyscale
variant that is 1 x 18 x 16 = 288 weights per neuron mapping the current frame to the current
firing rate — a single achromatic spatial filter per neuron, and no temporal integration at all.
That is also the ceiling on what this model class can do, however it is regularized.

The colour model has two such filters per neuron (green and UV). Here `DummyCore` averages the
channels before the readout ever sees them, so the montage below has one panel instead of two and
the per-channel norm plot has a single bar.

The smoothness penalty (`smooth_weight`) acts directly on these filters via a spatial Laplacian, so
this is where the regularized and unregularized arms visibly differ: at the shipped default of
`1.0` the penalty is ~1e-4 of the loss and the filters stay speckled; at `3e4` they are spatially
coherent. `LaplaceL2norm` is the *ratio* `sum(laplace(w)^2) / sum(w^2)`, hence channel-count
invariant — which is why the colour model's best weight transferred unchanged to this half-size
model.

In [ ]:
top_neurons = np.argsort(feve_score)[::-1][:3]

for rank, neuron in enumerate(top_neurons):
    # Lay the three panels out up front and hand the first two to the readout's plotting helper.
    # Left to itself it makes a 1x2 figure, so a 4th-of-4 subplot added afterwards lands on top of
    # the second panel instead of to the right of it.
    fig, (ax_readout, ax_features, resp_ax) = plt.subplots(
        ncols=3, figsize=(16, 4), gridspec_kw={"width_ratios": [1, 1, 2]}
    )
    model.readout[example_session].plot_weight_for_neuron(int(neuron), axes=(ax_readout, ax_features))

    resp_ax.plot(np.arange(lag, lag + window), mean_test_responses[neuron, lag:][:window], label="target")
    resp_ax.plot(np.arange(lag, lag + window), model_predictions[:window, neuron], label="prediction")
    resp_ax.set_xlabel("frame")
    resp_ax.set_ylabel("response")
    resp_ax.set_title(f"prediction (correlation {correlations[neuron]:.2f})")
    resp_ax.legend()
    sns.despine(ax=resp_ax)

    fig.suptitle(f"{example_session} neuron {neuron} (rank {rank + 1}, FEVe {feve_score[neuron]:.2f})")
    fig.tight_layout()
    plt.show()

## Getting the kernels out as numbers

The montage above is drawn straight from `readout.inner_product_kernel.weight` (see
`_plot_weight_for_neuron` in `openretina/modules/readout/linear_nonlinear_poison.py`). Often you
want that array rather than a picture of it — to compare filters across neurons, relate them to
cell types or ROI positions, or fit something on top. This pulls the filters for *every* neuron of
one session at once.


In [ ]:
def get_session_kernels(session_key: str) -> np.ndarray:
    """The learned spatial filters of every neuron in one session.

    For an LNP the readout kernel *is* the model: `inner_product_kernel` is an `nn.Conv3d` holding
    one filter per neuron, shaped (neurons, channels, 1, height, width) — the same weights
    `_plot_weight_for_neuron` draws. The singleton time axis (the model sees only the current
    frame) is squeezed out here, so one neuron's filter is a plain (channels, height, width) image
    in stimulus pixel space: (1, 18, 16) for this greyscale model, (2, 18, 16) for the colour one.

    Looks the notebook's current `model` up at call time, so it follows a re-run of the loading
    cell with a different `MODEL`.
    """
    if session_key not in model.readout:
        raise KeyError(
            f"{session_key!r} is not one of the model's {len(model.readout.keys())} sessions "
            f"(e.g. {sorted(model.readout.keys())[0]!r})"
        )
    weight = model.readout[session_key].inner_product_kernel.weight  # (neurons, ch, 1, h, w)
    return weight.detach().cpu().numpy()[:, :, 0]


# Any key of `model.readout`. Keeping it at `example_session` means the kernels line up row-for-row
# with the `feve_score` and `correlations` computed above, so they can be compared per neuron.
SESSION = example_session

kernels = get_session_kernels(SESSION)
kernel_norms = np.linalg.norm(kernels.reshape(len(kernels), -1), axis=1)

print(f"{SESSION}")
print(f"  kernels            {kernels.shape} (neurons, channels, height, width), {kernels.dtype}")
print(f"  weights per neuron {kernels[0].size}")
print(f"  weight range       [{kernels.min():+.4f}, {kernels.max():+.4f}]")
print(
    f"  L2 norm per neuron min {kernel_norms.min():.3f}, "
    f"median {np.median(kernel_norms):.3f}, max {kernel_norms.max():.3f}"
)

# The readout is exp(w . x + b), so the filters only make sense as rates alongside the bias.
bias = model.readout[SESSION].inner_product_kernel.bias
print(f"  bias               {'none' if bias is None else tuple(bias.shape)}")

---

## Evaluating a different model

Both LNP variants are published, so you can swap `MODEL` for either name and re-run from *Loading
the trained model* — the data above does not need reloading:

| `MODEL` | channels | weights/neuron | test corr |
|---|---|---|---|
| `hoefling_2024_lnp_low_res_grey_scale` | 1 | 288 | 0.2608 |
| `hoefling_2024_lnp_low_res` | 2 | 576 | 0.2652 |

Test correlation is the pooled per-neuron value over 3144 neurons / 67 sessions, from
`data_info["pretrained_performance"]`. Both were trained with `readout.smooth_weight=3e4`; at the
shipped default of `1.0` the penalty is ~1e-4 of the loss and does nothing.

**Loading the colour model here will fail on purpose.** The greyscale check above asserts one
readout channel, so a colour checkpoint trips it rather than quietly producing plausible-looking
results. Use `eval_hoefling_2024_lnp.ipynb` for the colour model — it composes the colour config.

`load_core_readout_model` also takes local paths, so an in-progress training run can be evaluated
the same way; see the commented alternative in the loading cell. Runs from
`run_hoefling_lnp_grey.sh` are laid out as
`openretina_assets/runs/lnp_grey/<jobid>/<arm>/checkpoints/`.